## Загрузка модели с HuggingFace

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
def download_qwen_thinking():
    model_name = "Qwen/Qwen3-4B-Instruct-2507"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map=None
    ).to(DEVICE)

    model.eval()
    return model, tokenizer

In [ ]:
model, tokenizer = download_qwen_thinking()

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

## Дополнение запросов + исправление опечаток

In [ ]:
def is_meaningful_query(query: str) -> bool:
    """
    Возвращает True, если запрос похож на реальный поисковый запрос
    """
    query = query.strip()

    if not query:
        return False

    if len(query) < 2:
        return False

    if re.match(r'^(.)\1{3,}$', query):  # 3+ одинаковых символов
        return False

    if re.match(r'^[\.\,\!\?\-\_\=\+\*\#\@\$\%\^\&\;]+$', query):
        return False

    if re.match(r'^[0-9\s]+$', query):
        return False

    return True

In [ ]:
import json
import re

def generate_synonyms(model, tokenizer, query: str):
    """Генерация синонимов для Instruct-модели"""
    query = query.strip()
    if not is_meaningful_query(query):
        return query, [query] if query else ["не указан"]


    prompt = f"""<|im_start|>user
Исправь опечатки и придумай 3 разных синонима для поиска товара: "{query}"

Правила:
1. Синонимы должны быть РАЗНЫМИ по формулировке. Не повторяй одно и то же.
2. Если запрос похож на русское слово, набранное в английской раскладке (например "ghjcnfr" вместо "принтер", "en.u" вместо "утюг") — переведи в русскую раскладку.
3. Для технических терминов ("флешка", "SSD") используй разные формулировки.

Ответь только JSON в формате: {{"corrected": "...", "variants": ["...", "...", "..."]}}
<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs['input_ids'].shape[1]
    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    start = response.find('{')
    end = response.rfind('}')

    if start != -1 and end != -1:
        json_str = response[start:end+1]
        result = json.loads(json_str)
        corrected = result.get("corrected", query)
        variants = result.get("variants", [query])
    else:
        corrected = query
        variants = [query]

    return corrected, variants

## Тесты

In [ ]:
def test_gen_synonims(query):
    corrected, variants = generate_synonyms(model, tokenizer, query)
    print(corrected)
    print(variants)

In [ ]:
def test_gen_synonims_for_list(queries):
    for query in queries:
        print(f"Query: {query}")
        test_gen_synonims(query)

In [ ]:
test_gen_synonims("бумага А4")

бумага А4
['бумага формата А4', 'лист А4', 'бумага 210x297']


In [ ]:
# тестовый набор №1
# нормальные запросы
normal_queries = [
    "бумага А4",
    "принтер лазерный",
    "картридж HP",
    "клавиатура компьютерная",
    "мышь беспроводная",
    "монитор 24 дюйма",
    "стол офисный",
    "кресло компьютерное",
    "флешка 32GB",
    "наушники с микрофоном",
]

# запросы с опечатками
typo_queries = [
    "прінтер лазерный",
    "картриж HP",
    "кресло компютерное",
    "блк питания для компьютера",
    "сенсорная мыш",
    "конфеты с конбяком",
    "сотл офисный"
]

# запросы с сокращениями слов/сленгами/просторечиями
synonym_queries = [
    "лазерник",                     # сленг: лазерный принтер
    "струйник",                     # сленг: струйный принтер
    "расходник для принтера",
    "канцелярия",                   # общее понятие
    "оргтехника",
    "периферия для компа",
    "мыша",                         # разговорное
    "клава",                        # сленг: клавиатура
    "системник",                    # системный блок
    "материнка",                    # материнская плата
    "оперативка",                   # оперативная память
    "зарядка для ноута",            # блок питания ноутбука
    "кулер процессора",             # охлаждение
    "бп компьютерный",              # блок питания
    "вебка",                        # веб-камера
    "б/у принтер",
]

# сложные/составные запросы, для проверки сохранения подробностей о товаре
complex_queries = [
    "принтер лазерный черно-белый А4 с автоподачей",
    "бумага офисная А4 80г/м² плотность белизна 146%",
    "картридж HP 650 совместимый с LaserJet Pro",
    "монитор 27 дюймов IPS 4K HDR10",
    "игровая клавиатура механическая с подсветкой",
    "беспроводная мышь Logitech MX Master 3S",
    "накопитель SSD M.2 NVMe 1TB",
    "ноутбук для работы и учебы 15 дюймов",
    "МФУ лазерное с Wi-Fi и сканером",
    "проектор Full HD для презентаций",
]

# неадекватные запросы
insane_queries = [
    "",
    " ",
    "авыало",
    "ывапаро",
    "............",
    "принтер ######",
    "1234567890",
    "йцукенгшщзхъ",
    "фывапролджэ",
    "printer hp",
    "бумага А4 ............",
    "en.u" # утюг, чел не поменял раскладку
    "принтер принтер принтер",
    "очень длинный запрос про принтер лазерный который нужен для офиса чтобы печатать документы ежедневно",
    "а",
    "яяяяяяя",
    "смешарики",
    "погода в москве",
    "где купить дешёвый принтер",
    "какой принтер лучше",
    "скидка на бумагу",
    "принтер -новый",
]

In [ ]:
test_gen_synonims_for_list(normal_queries)

Query: бумага А4
бумага А4
['бумага формата А4', 'бумага 210x297', 'офисная бумага А4']
Query: принтер лазерный
лазерный принтер
['лазерный принтер', 'принтер лазерный', 'ч/б лазерный принтер']
Query: картридж HP
картридж HP
['картридж для HP', 'тонер HP', 'тонер для HP']
Query: клавиатура компьютерная
клавиатура компьютерная
['клавиатура для компьютера', 'компьютерная клавиатура', 'безынтерфейсная клавиатура']
Query: мышь беспроводная
мышь беспроводная
['беспроводная мышь', 'мышь без проводов', 'мышь с беспроводным соединением']
Query: монитор 24 дюйма
монитор 24 дюйма
['монитор 24 дюйма', 'дисплей 24 дюйма', 'экран 24 дюйма']
Query: стол офисный
стол офисный
['офисный стол', 'стол для офиса', 'стол в офисе']
Query: кресло компьютерное
кресло компьютерное
['компьютерное кресло', 'кресло для работы', 'офисное кресло']
Query: флешка 32GB
флешка 32 ГБ
['флешка 32 гигабайта', '32 гб флешка', 'флешка на 32 гб']
Query: наушники с микрофоном
наушники с микрофоном
['наушники с микрофоном', 'н

In [ ]:
test_gen_synonims_for_list(typo_queries)

Query: прінтер лазерный
принтер лазерный
['лазерный принтер', 'принтер для лазерной печати', 'чёрно-белый лазерный принтер']
Query: картриж HP
картридж HP
['картридж для HP', 'тонер HP', 'запчасть для HP']
Query: кресло компютерное
кресло компьютерное
['компьютерное кресло', 'кресло для компьютера', 'офисное кресло']
Query: блк питания для компьютера
блок питания для компьютера
['блок питания для ПК', 'питание для компьютера', 'блоки питания для ПК']
Query: сенсорная мыш
сенсорная мышь
['мышь сенсорная', 'сенсорная клавиатура', 'мышь с сенсором']
Query: конфеты с конбяком
конфеты с кокосом
['конфеты с кокосом', 'кокосовые конфеты', 'конфеты с добавлением кокоса']
Query: сотл офисный
сотовый офисный
['офисный сотовый', 'сотовое устройство для офиса', 'офисный телефон']


In [ ]:
test_gen_synonims_for_list(synonym_queries)

Query: лазерник
лазерный принтер
['лазерный принтер', 'принтер лазерный', 'ч/б принтер']
Query: струйник
струйный принтер
['принтер струйный', 'струйный принтер для дома', 'принтер для печати на бумаге']
Query: расходник для принтера
расходник для принтера
['запасные части для принтера', 'тонер для принтера', 'папки для принтера']
Query: канцелярия
канцелярия
['канцелярские товары', 'товары для офиса', 'офисная канцелярия']
Query: оргтехника
оргтехника
['офисная техника', 'оборудование для офиса', 'техника для бизнеса']
Query: периферия для компа
периферия для компьютера
['оборудование для компьютера', 'аксессуары для ПК', 'устройства для компьютера']
Query: мыша
мышь
['мышь для компьютера', 'устройство мыши', 'мышь компьютерная']
Query: клава
клавиатура
['клавиатура', 'клава', 'клавиатура для компьютера']
Query: системник
системник
['системный блок', 'комплект системы', 'система компьютерная']
Query: материнка
материнская плата
['материнская плата', 'платформа для ПК', 'мать компьютер

In [ ]:
test_gen_synonims_for_list(complex_queries)

Query: принтер лазерный черно-белый А4 с автоподачей
лазерный принтер черно-белый А4 с автоподачей
['лазерный ч/б принтер А4', 'черно-белый лазерный принтер А4', 'принтер для офиса лазерный ч/б']
Query: бумага офисная А4 80г/м² плотность белизна 146%
бумага офисная А4 80 г/м² плотность белизна 146%
['бумага А4 80 г/м²', 'офисная бумага 80 г/м²', 'бумага для печати А4']
Query: картридж HP 650 совместимый с LaserJet Pro
картридж HP 650 совместимый с LaserJet Pro
['картридж для LaserJet Pro', 'совместимый картридж HP 650', 'запасной картридж для HP 650']
Query: монитор 27 дюймов IPS 4K HDR10
монитор 27 дюймов IPS 4K HDR10
['монитор 27 дюймов IPS 4К HDR10', 'дисплей 27 дюймов IPS 4K с поддержкой HDR10', 'монитор 4K 27 дюймов IPS с HDR10']
Query: игровая клавиатура механическая с подсветкой
игровая механическая клавиатура с подсветкой
['механическая игровая клавиатура с подсветкой', 'клавиатура для игр с подсветкой', 'подсвеченная механическая клавиатура для игр']
Query: беспроводная мышь L

In [ ]:
test_gen_synonims_for_list(insane_queries)

Query: 
принтер лазерный
['лазерный принтер', 'принтер для офиса', 'ч/б принтер']
Query:  
принтер лазерный
['лазерный принтер', 'принтер для офиса', 'ч/б принтер']
Query: авыало
аввало
['аввало', 'аввало', 'аввало']
Query: ывапаро
выпаро
['выпариватель', 'пароварка', 'пароварка для воды']
Query: ............
принтер лазерный
['лазерный принтер', 'принтер для офиса', 'ч/б принтер']
Query: принтер ######
принтер лазерный
['лазерный принтер', 'принтер для офиса', 'ч/б принтер']
Query: 1234567890
1234567890
['1234567890', '1234567890 товар', 'товар 1234567890']
Query: йцукенгшщзхъ
йцукенгшщзхъ
['йцукенгшщзхъ', 'йцукенгшщзх', 'йцукенгшщз']
Query: фывапролджэ
фывапролджэ
['фывапролджэ', 'фывапролджэ', 'фывапролджэ']
Query: printer hp
printer hp
['hp принтер', 'принтер hp', 'принтер от hp']
Query: бумага А4 ............
бумага А4
['бумага формата А4', 'бумага 210x297 мм', 'бумага для принтера А4']
Query: en.uпринтер принтер принтер
en.упринтер принтер
['принтер для дома', 'без бумаги принтер

## Тесты №2

In [ ]:
# нормальные запросы
normal_queries = [
    "бумага А4",
    "принтер лазерный",
    "картридж HP",
    "клавиатура компьютерная",
    "мышь беспроводная",
    "монитор 24 дюйма",
    "стол офисный",
    "кресло компьютерное",
    "флешка 32GB",
    "наушники с микрофоном",
    "Samsung UE32H5000FUXRU (2025), LED",
    "HP LaserJet 1020",
    "SSD M.2 NVMe 1TB",
]

# Запросы с опечатками
typo_queries = [
    "прнтер лазерный",
    "кресло компютерное",
    "блк питания для компьютера",
    "яболчный сок",
    "сутл офисный",
    "маттеринска плата",
    "процессор интэл",
]

# Запросы с сокращениями/сленгом
slang_queries = [
    "лазерник",                     # сленг: лазерный принтер
    "струйник",                     # сленг: струйный принтер
    "расходник для принтера",
    "канцелярия",
    "оргтехника",
    "периферия для компа",
    "мыша",
    "клава",
    "системник",
    "материнка",
    "оперативка",
    "зарядка для ноута",
    "кулер процессора",
    "бп компьютерный",
    "вебка",
    "б/у принтер",
]

# Сложные составные запросы
complex_queries = [
    "принтер лазерный черно-белый А4 с автоподачей",
    "бумага офисная А4 80г/м² плотность белизна 146%",
    "картридж HP 650 совместимый с LaserJet Pro",
    "монитор 27 дюймов IPS 4K HDR10",
    "игровая клавиатура механическая с подсветкой",
    "беспроводная мышь Logitech MX Master 3S",
    "накопитель SSD M.2 NVMe 1TB",
    "ноутбук для работы и учебы 15 дюймов",
    "МФУ лазерное с Wi-Fi и сканером",
    "проектор Full HD для презентаций",
]

# Бессмысленные запросы
meaningless_queries = [
    "",
    " ",
    "а",
    "шщреоол",
    "ххххы",
    "............",
    "принтер######",
    "78246",
    "фывапролджэ",
    "яяяяяяя",                      # повтор символа
    "gsktcjc",                         # пылесос
    "принтер принтер принтер",      # повтор слова (не бессмысленный, но странный)
    "кроватка детск4ая",
    "где купить дешёвый принтер",
    "какой принтер лучше",
    "скидка на бумагу",
    "смешарики",
    "погода в москве",
    "сколько стоит принтер",
    "доставка принтера",
]


In [ ]:
test_gen_synonims_for_list(normal_queries)

Query: бумага А4
бумага А4
['бумага формата А4', 'лист А4', 'бумага 210x297 мм']
Query: принтер лазерный
принтер лазерный
['лазерный принтер', 'принтер на основе лазера', 'лазерная печать для дома']
Query: картридж HP
картридж HP
['тонер HP', 'запасной картридж для HP', 'контейнер для чернил HP']
Query: клавиатура компьютерная
клавиатура компьютерная
['клавиатура для ПК', 'безпроводная клавиатура для компьютера', 'компьютерная клавиатура с подсветкой']
Query: мышь беспроводная
мышь беспроводная
['беспроводная мышь', 'мыши без проводов', 'мышь без кабеля']
Query: монитор 24 дюйма
монитор 24 дюйма
['дисплей 24 дюйма', 'экран 24 дюйма', 'монитор размером 24 дюйма']
Query: стол офисный
стол офисный
['офисный стол', 'стол для офиса', 'рабочий стол']
Query: кресло компьютерное
кресло для компьютера
['стол для работы за компьютером', 'диван для письма за компьютером', 'шасси для компьютера']
Query: флешка 32GB
флешка 32 ГБ
['флешка на 32 гигабайта', 'USB-накопитель 32 ГБ', 'накопитель на 32 Г

In [ ]:
test_gen_synonims_for_list(typo_queries)

Query: прнтер лазерный
принтер лазерный
['лазерный принтер', 'принтер лазерного типа', 'лазерный устройство для печати']
Query: кресло компютерное
кресло компьютерное
['компьютерное кресло', 'кресло для компьютера', 'стол для работы с компьютером']
Query: блк питания для компьютера
блок питания для компьютера
['блок подключения для ПК', 'питание для ноутбука и ПК', 'блоки питания для компьютера и периферии']
Query: яболчный сок
яблочный сок
['сок из яблок', 'яблочный сок для дома', 'сок яблочный натуральный']
Query: сутл офисный
свиток офисный
['офисный свиток', 'свиток для офиса', 'офисное устройство свиток']
Query: маттеринска плата
материнская плата
['платформа для процессора', 'мать компьютера', 'основная плата']
Query: процессор интэл
процессор intel
['процессор от intel', 'интэл процессор', 'процессор серии intel']


In [ ]:
test_gen_synonims_for_list(slang_queries)

Query: лазерник
лазерный принтер
['принтер лазерный', 'лазерный принтер для дома', 'устройство для лазерной печати']
Query: струйник
струйный принтер
['принтер струйный', 'струйная печать', 'принтер с чернильным картриджем']
Query: расходник для принтера
расходник для принтера
['запасные части для принтера', 'тонер и чернила для принтера', 'компоненты для принтера']
Query: канцелярия
канцелярия
['канцелярские товары', 'товары для офиса', 'офисная принадлежность']
Query: оргтехника
оргтехника
['оборудование для офиса', 'официальная техника', 'техника для работы']
Query: периферия для компа
периферийные устройства для компьютера
['устройства для ПК', 'внешние аксессуары для ноутбука', 'аксессуары для компьютера']
Query: мыша
мышь
['мышь для компьютера', 'устройство для мыши', 'мышь компьютерная']
Query: клава
клавиатура
['клавиатура', 'клавиатура для компьютера', 'клава для ПК']
Query: системник
системник
['системный блок', 'комплексное решение', 'оборудование для системы']
Query: матери

In [ ]:
test_gen_synonims_for_list(complex_queries)

Query: принтер лазерный черно-белый А4 с автоподачей
лазерный принтер черно-белый А4 с автоподачей
['черно-белый лазерный принтер А4 с автоматической подачей', 'лазерный принтер А4 без цвета с автоподачей листов', 'принтер на базе лазерной технологии А4 черно-белый с автоматической подачей']
Query: бумага офисная А4 80г/м² плотность белизна 146%
бумага офисная А4 80 г/м² плотность белизна 146%
['офисная бумага А4 80 г/м² с белизной 146%', 'бумага А4 80 г/м² для офиса с показателем белизны 146%', 'бумага офисная 80 г/м² А4 высокая белизна 146%']
Query: картридж HP 650 совместимый с LaserJet Pro
картридж HP 650 совместимый с LaserJet Pro
['картридж для принтера HP 650 совместимый с LaserJet Pro', 'совместимый картридж HP 650 для LaserJet Pro', 'замена картриджу HP 650 для принтера LaserJet Pro']
Query: монитор 27 дюймов IPS 4K HDR10
монитор 27 дюймов IPS 4К HDR10
['27-дюймовый монитор с экраном IPS и разрешением 4K HDR10', 'монитор 27 дюймов 4К с технологией IPS и поддержкой HDR10', 'мон

In [ ]:
test_gen_synonims_for_list(meaningless_queries)

Query: 

['не указан']
Query:  

['не указан']
Query: а
а
['а']
Query: шщреоол
штормол
['штормол', 'штормол', 'штормол']
Query: ххххы
ххххы
['ххххы']
Query: ............
............
['............']
Query: принтер######
принтер######
['принтер######']
Query: 78246
78246
['78246']
Query: фывапролджэ
фывапролджэ
['флешка', 'накопитель', 'USB-устройство']
Query: яяяяяяя
яяяяяяя
['яяяяяяя']
Query: gsktcjc
gsktcjc
['gskt cjc', 'gsktcjc поиск', 'gsktcjc товар']
Query: принтер принтер принтер
принтер
['устройство для печати', 'машинка для печати', 'печать документа']
Query: кроватка детск4ая
кроватка детская
['детская кроватка', 'кроватка для детей', 'кроватка для малыша']
Query: где купить дешёвый принтер
где купить дешёвый принтер
['где найти недорогой принтер', 'как купить бюджетный принтер', 'где купить принтер по低价']
Query: какой принтер лучше
какой принтер лучше
['лучший принтер для дома', 'на что посмотреть при покупке принтера', 'какой принтер стоит выбрать']
Query: скидка на бумагу


## Получение финальных товаров и расчет нмцк

Мой воображаемый пайплайн

найдено n товаров

1. ДЕДУПЛИКАЦИЯ
   Группируем одинаковые товары из разных источников, удаляем явные дубликаты (один и тот же продавец и товар на разных площадках, берем медианное значение чтоб сохранить правдивое распределение на рынке)

    ↓
2. РЕРАНКЕР (по релевантности к исправленному запросу)
   Сортируем уникальные товары по релевантности
   Оставляем например топ-20

    ↓
3. ФИЛЬТРАЦИЯ
   Отбрасываем цены, отклоняющиеся от среднего >33% Повторяем итеративно

    ↓
4. РАСЧЁТ НМЦК = среднее арифметическое оставшихся цен

    ↓
Если осталось <5 источников/товаров → фолбэк


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

class PriceAnalyzer:
    def __init__(self):
        self.embedder = SentenceTransformer('BAAI/bge-m3')
        self.reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')

    def deduplicate(self, products: list[dict], threshold: float = 0.95) -> list[dict]:
        """
        Группирует одинаковые товары через косинусную близость эмбеддингов
        """
        if not products:
            return []

        titles = [p.get('title', '') for p in products]
        embeddings = self.embedder.encode(titles, normalize_embeddings=False)

        used = set()
        unique_products = []

        for i in range(len(products)):
            if i in used:
                continue

            group = [products[i]]
            used.add(i)

            for j in range(i + 1, len(products)):
                if j in used:
                    continue
                cos_sim = cosine_similarity([embeddings[i]], [embeddings[j]])[0][0]

                if cos_sim > threshold:
                    group.append(products[j])
                    used.add(j)

            if len(group) > 1:
                print(f"Дубликат: '{group[0]['title']}'")
                for dup in group[1:]:
                    print(f"  '{dup['title']}' (совпадение)")

            prices = [p.get('price', float('inf')) for p in group]
            median_price = np.median(prices)

            representative = group[0].copy()
            representative['price'] = round(median_price, 2)
            representative['sources_count'] = len(group)
            representative['source_names'] = [p.get('source', 'unknown') for p in group]

            unique_products.append(representative)

        return unique_products


    def rerank(self, query: str, products: list[dict], top_k: int = 20) -> list[dict]:
        """
        Сортирует товары по релевантности запросу
        """
        if not products:
            return []

        pairs = [(query, p.get('title', '')) for p in products]
        scores = self.reranker.predict(pairs)

        for product, score in zip(products, scores):
            product['relevance_score'] = float(score)

        sorted_products = sorted(products, key=lambda x: x['relevance_score'], reverse=True)

        return sorted_products[:top_k]

    def filter_outliers(self, prices: list[float], max_deviation: float = 0.33) -> list[float]:
        """
        Фильтрация по методике 44-ФЗ п.3.20 (отбрасываются цены, отклоняющиеся от среднего >33%)
        """
        if len(prices) < 3:
            return prices

        mean_price = np.mean(prices)

        filtered = []
        for price in prices:
            deviation = abs(price - mean_price) / mean_price
            if deviation <= max_deviation:
                filtered.append(price)

        # Если после фильтрации осталось меньше 5 цен — возвращаем оригинал
        if len(filtered) < 5:
            return prices

        # Рекурсивно проверяем ещё раз (могли появиться новые выбросы)
        if len(filtered) != len(prices):
            return self.filter_outliers(filtered, max_deviation)

        return filtered


    def select_top_n_prices(self, prices: list[float], target_count: int = 5) -> list[float]:
        """
        Используется, когда после фильтрации осталось больше 5 товаров, выбирает 5 цен, наиболее близких к среднему значению.
        """
        if len(prices) <= target_count:
            return prices

        mean_price = np.mean(prices)
        sorted_by_closeness = sorted(prices, key=lambda x: abs(x - mean_price))

        return sorted_by_closeness[:target_count]


    def calculate_nmck(self, products: list[dict]) -> dict:
        """
        Расчёт НМЦК с отбором ровно 5 товаров
        """
        if not products:
            return {
                'nmck': None,
                'price_count': 0,
                'all_prices': [],
                'min_price': None,
                'max_price': None,
                'status': 'no_products'
            }

        prices = [p['price'] for p in products if p.get('price')]

        if len(prices) < 5:
            return {
                'nmck': np.mean(prices) if prices else None,
                'price_count': len(prices),
                'all_prices': prices,
                'min_price': min(prices) if prices else None,
                'max_price': max(prices) if prices else None,
                'status': 'insufficient_data'
            }

        filtered_prices = self.filter_outliers(prices)

        # Если после фильтрации осталось меньше 5 товаров — ошибка
        if len(filtered_prices) < 5:
            return {
                'nmck': None,
                'price_count': len(prices),
                'filtered_count': len(filtered_prices),
                'all_prices': prices,
                'filtered_prices': filtered_prices,
                'min_price': min(prices),
                'max_price': max(prices),
                'status': 'filtered_too_much'
            }

        # Если осталось больше 5 — берём 5 ближайших к среднему
        if len(filtered_prices) > 5:
            filtered_prices = self.select_top_n_prices(filtered_prices, 5)

        # Проверяем условие <33% для отобранных цен (нужно убедиться, что в финальном наборе нет сильного разброса)
        mean_price = np.mean(filtered_prices)
        deviations = [abs(p - mean_price) / mean_price for p in filtered_prices]
        max_deviation = max(deviations)

        if max_deviation > 0.33:
            return {
                'nmck': None,
                'price_count': len(prices),
                'filtered_count': len(filtered_prices),
                'all_prices': prices,
                'filtered_prices': filtered_prices,
                'min_price': min(filtered_prices),
                'max_price': max(filtered_prices),
                'max_deviation': round(max_deviation, 3),
                'status': 'warning_deviation_too_high'
            }

        nmck = np.mean(filtered_prices)

        return {
            'nmck': round(nmck, 2),
            'price_count': len(prices),
            'filtered_count': len(filtered_prices),
            'all_prices': prices,
            'filtered_prices': filtered_prices,
            'min_price': min(filtered_prices),
            'max_price': max(filtered_prices),
            'status': 'success'
        }

    def process(self, query: str, products: list[dict]) -> dict:
        """
        Полный пайплайн обработки
        """
        # Шаг 1: Дедупликация
        print(f"Исходно товаров: {len(products)}")
        unique_products = self.deduplicate(products)
        print(f"После дедупликации: {len(unique_products)}")

        # Шаг 2: Реранкер
        ranked_products = self.rerank(query, unique_products, top_k=20)
        print(f"После реранкера (топ-20): {len(ranked_products)}")

        # Шаг 3: Расчёт НМЦК
        result = self.calculate_nmck(ranked_products)

        result['products'] = ranked_products[:10]
        result['query'] = query

        return result

def handle_insufficient_data(result: dict, query: str) -> dict:
    """
    Обработка случаев, когда нельзя рассчитать НМЦК
    """
    status = result.get('status')

    if status == 'no_products':
        return {
            **result,
            'message': 'Товары не найдены. Попробуйте изменить запрос.',
            'recommendation': 'расширить_запрос',
            'can_calculate_nmck': False
        }

    if status == 'insufficient_data':
        return {
            **result,
            'message': f'Найдено только {result["price_count"]} товаров. Для расчёта НМЦК требуется минимум 5 товаров.',
            'recommendation': 'расширить_запрос_или_добавить_источники',
            'can_calculate_nmck': False
        }

    if status == 'filtered_too_much':
        return {
            **result,
            'message': f'После отсева выбросов осталось только {result["filtered_count"]} товаров из {result["price_count"]}. Разброс цен слишком большой (от {result["min_price"]} до {result["max_price"]} ₽).',
            'recommendation': 'добавить_источники_или_расширить_поиск',
            'can_calculate_nmck': False
        }

    if status == 'warning_deviation_too_high':
        return {
            **result,
            'message': f'Разброс финальных цен превышает 33% (максимальное отклонение: {result["max_deviation"]*100:.1f}%). НМЦК не рассчитана, рекомендуется добавить больше источников для уточнения.',
            'recommendation': 'добавить_источники',
            'can_calculate_nmck': False,
        }

    return {
        **result,
        'message': f'НМЦК успешно рассчитана на основе {result["filtered_count"]} цен.',
        'recommendation': None,
        'can_calculate_nmck': True,
        'nmck': result.get('nmck')
    }



## Пример работы получения

In [ ]:
analyzer = PriceAnalyzer()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
products = [
    # Принтеры (релевантные)
    {'title': 'Принтер HP LaserJet 1020', 'price': 12500, 'source': 'ozon'},
    {'title': 'HP LaserJet 1020 принтер', 'price': 11800, 'source': 'wb'},
    {'title': 'Принтер Canon LBP 6030', 'price': 13200, 'source': 'yandex'},
    {'title': 'Лазерный принтер Pantum P2500W', 'price': 12800, 'source': 'ozon'},
    {'title': 'Принтер Xerox B210', 'price': 13500, 'source': 'wb'},
    {'title': 'Brother HL-1210WR принтер', 'price': 12200, 'source': 'yandex'},
    {'title': 'Принтер Kyocera FS-1040', 'price': 13000, 'source': 'ozon'},
    {'title': 'Лазерный принтер Ricoh SP 210', 'price': 12700, 'source': 'wb'},
    {'title': 'Принтер Samsung ML-2160', 'price': 12500, 'source': 'yandex'},
    {'title': 'Принтер HP Neverstop 1000', 'price': 14000, 'source': 'ozon'},

    # Выбросы (дешёвые б/у)
    {'title': 'Принтер HP LaserJet 1020 б/у', 'price': 4500, 'source': 'runet'},
    {'title': 'Лазерный принтер б/у', 'price': 3800, 'source': 'runet'},

    # Выбросы (дорогие)
    {'title': 'Принтер HP LaserJet 1020 новый в упаковке', 'price': 25000, 'source': 'ozon'},
    {'title': 'Принтер с доставкой', 'price': 30000, 'source': 'wb'},

    # Совсем нерелевантные (должны отсеяться)
    {'title': 'Бумага А4 Снегурочка', 'price': 350, 'source': 'ozon'},
    {'title': 'Картридж HP 650', 'price': 2500, 'source': 'wb'},
    {'title': 'Клавиатура Logitech K380', 'price': 4000, 'source': 'yandex'},
    {'title': 'Мышь беспроводная', 'price': 1200, 'source': 'ozon'},
    {'title': 'Наушники с микрофоном', 'price': 3000, 'source': 'wb'},
    {'title': 'Чехол для ноутбука', 'price': 1500, 'source': 'runet'},
    {'title': 'Стул офисный', 'price': 5000, 'source': 'ozon'},
    {'title': 'Лампа настольная', 'price': 2000, 'source': 'wb'},
    {'title': 'Флешка 64GB', 'price': 800, 'source': 'yandex'},
    {'title': 'Внешний жесткий диск 1TB', 'price': 5500, 'source': 'ozon'},
    {'title': 'МФУ Canon', 'price': 18000, 'source': 'wb'},
    {'title': 'Сканер Epson', 'price': 7500, 'source': 'yandex'},
    {'title': 'Проектор Epson', 'price': 22000, 'source': 'ozon'},
    {'title': 'Доска магнитно-маркерная', 'price': 3500, 'source': 'wb'},
    {'title': 'Швабра с отжимом', 'price': 1200, 'source': 'yandex'},
]

corrected_query = "принтер лазерный"

result = analyzer.process(corrected_query, products)

result = handle_insufficient_data(result, corrected_query)


print(f"\nЗапрос: {result['query']}")
print(f"Всего товаров: {result['price_count']}")
print(f"Отобрано для НМЦК: {result.get('filtered_count', 0)}")
print(f"НМЦК: {result['nmck']} ₽")
print(f"Мин. цена: {result['min_price']} ₽")
print(f"Макс. цена: {result['max_price']} ₽")
print(f"Статус: {result['status']}")
print(f"Сообщение: {result.get('message', 'Нет')}")
print(f"Можно рассчитать: {result.get('can_calculate_nmck', False)}")

print("ТОП-10 ТОВАРОВ ПОСЛЕ РЕРАНКЕРА")

for i, p in enumerate(result.get('products', []), 1):
    print(f"{i}. [{p['source']}] {p['title']} — {p['price']} ₽ (релевантность: {p.get('relevance_score', 0):.3f})")

print("ЦЕНЫ, ПОПАВШИЕ В РАСЧЁТ НМЦК")

if result.get('filtered_prices'):
    for price in result['filtered_prices']:
        print(f"  • {price} ₽")
else:
    print("Нет данных для расчёта")

print("ВСЕ ЦЕНЫ (ДО ФИЛЬТРАЦИИ)")

if result.get('all_prices'):
    for price in sorted(result['all_prices']):
        print(f"  • {price} ₽")

Исходно товаров: 29
Дубликат: 'Принтер HP LaserJet 1020'
  'HP LaserJet 1020 принтер' (совпадение)
После дедупликации: 28
После реранкера (топ-20): 20

Запрос: принтер лазерный
Всего товаров: 20
Отобрано для НМЦК: 5
НМЦК: 12840.0 ₽
Мин. цена: 12500.0 ₽
Макс. цена: 13200.0 ₽
Статус: success
Сообщение: НМЦК успешно рассчитана на основе 5 цен.
Можно рассчитать: True
ТОП-10 ТОВАРОВ ПОСЛЕ РЕРАНКЕРА
1. [ozon] Лазерный принтер Pantum P2500W — 12800.0 ₽ (релевантность: 0.975)
2. [wb] Лазерный принтер Ricoh SP 210 — 12700.0 ₽ (релевантность: 0.963)
3. [runet] Лазерный принтер б/у — 3800.0 ₽ (релевантность: 0.959)
4. [runet] Принтер HP LaserJet 1020 б/у — 4500.0 ₽ (релевантность: 0.708)
5. [ozon] Принтер HP LaserJet 1020 — 12150.0 ₽ (релевантность: 0.656)
6. [ozon] Принтер HP LaserJet 1020 новый в упаковке — 25000.0 ₽ (релевантность: 0.487)
7. [wb] Принтер с доставкой — 30000.0 ₽ (релевантность: 0.115)
8. [wb] МФУ Canon — 18000.0 ₽ (релевантность: 0.067)
9. [yandex] Brother HL-1210WR принтер — 1

Есть небольшая проблемка с низкой релевантностью норм запросов, типо:

10. [yandex] Принтер Canon LBP 6030 — 13200.0 ₽ (релевантность: 0.047)

потом подумаю как это решить
